# Mental Health LLM Baseline

Phase 1: GPT-4o (API) + Qwen2.5-7B (open-source)

---
## 0. Setup

In [ ]:
import os
if not os.path.exists('/content/8307-project'):
    !git clone https://github.com/zigeng22/8307-project.git /content/8307-project
    print('Repo cloned')
else:
    !cd /content/8307-project && git pull
    print('Repo updated')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# verify datasets on Drive
# upload to MyDrive/8307/Datasets/ first:
#   Combined Data.csv, medquad.csv, MentalChat16K-main/
import os
drive_data = '/content/drive/MyDrive/8307/Datasets'
assert os.path.exists(drive_data), f'Upload datasets to {drive_data} first'
print('Datasets:', os.listdir(drive_data))

In [ ]:
import os
link = '/content/Datasets'
src = '/content/drive/MyDrive/8307/Datasets'
if os.path.exists(link): os.remove(link)
os.symlink(src, link)
print(f'{link} -> {src}')

In [ ]:
!pip install -q transformers>=4.43.0 peft>=0.11.0 trl>=0.9.0 datasets accelerate
!pip install -q langchain langchain-community faiss-cpu sentence-transformers
!pip install -q rouge-score bert-score scikit-learn
!pip install -q openai anthropic tqdm pandas matplotlib seaborn

In [ ]:
import sys, os, torch
sys.path.insert(0, '/content/8307-project')
os.chdir('/content/8307-project')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('NO GPU')

In [ ]:
# paste your key when prompted (NOT saved to GitHub)
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Enter OpenRouter API Key: ')
from openai import OpenAI
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'])
r = client.chat.completions.create(model='openai/gpt-4o', messages=[{'role':'user','content':'Hi'}], max_tokens=5)
print('API OK:', r.choices[0].message.content)

In [ ]:
from data.loader import load_sentiment, load_mentalchat, load_medquad
from data.splitter import split_task1, split_task2, split_task3
df1 = load_sentiment(); print(f'Sentiment: {len(df1)}')
df2 = load_mentalchat(); print(f'MentalChat: {len(df2)}')
df3 = load_medquad(); print(f'MedQuAD: {len(df3)}')
test1 = split_task1(df1); print(f'Task1 test: {len(test1)}')
_, test2 = split_task2(df2); print(f'Task2 test: {len(test2)}')
test3 = split_task3(df3); print(f'Task3 test: {len(test3)}')
print('All data OK')

---
## 1. GPT-4o Baseline

In [ ]:
!python experiments/run_baseline.py --model gpt-4o --task 1

In [ ]:
!python experiments/run_baseline.py --model gpt-4o --task 2

In [ ]:
!python experiments/run_baseline.py --model gpt-4o --task 3

---
## 2. Qwen2.5-7B Baseline

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task 1

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task 2

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task 3

---
## 3. Results

In [ ]:
import json
from pathlib import Path
for d in sorted(Path('results/baseline').iterdir()):
    if d.is_dir():
        print(f'\n{d.name}:')
        for f in sorted(d.glob('*_metrics.json')):
            print(f'  {f.stem}: {json.loads(f.read_text())}')

In [ ]:
!mkdir -p /content/drive/MyDrive/8307/results_backup
!cp -r results/ /content/drive/MyDrive/8307/results_backup/
print('Backed up')